# Phase 3a - Input resolution ablation (Colab T4)

Trains each detector at **416 / 640 / 832** with everything else held fixed,
so the only variable is input resolution.

**Budget (measured from the baselines):** ~2.0 h child + ~2.3 h hazard at 30
epochs = **~4.3 h total**. Run **one model per session** — each is comfortably
inside a single Colab session.

**Resumable:** every finished resolution is appended to
`results/metrics/ablation_imgsz_<model>.csv` and skipped on re-run. Runs go to
Drive, so a disconnect costs at most the in-flight resolution.

**Caveat for the write-up:** 30 epochs is a comparison budget, not
convergence. Higher resolutions can need more epochs to pay off, so this may
understate 832. If 832 wins anyway, that's a strong result.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.4.106 roboflow

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
ABL_PROJECT = "/content/drive/MyDrive/deeplrn_group2/runs"

# Do NOT makedirs() blindly: if the folder is missing (wrong account, or a
# shared folder with no My Drive shortcut) that would silently create an
# EMPTY decoy and the run would retrain everything from scratch.
if os.path.isdir(ABL_PROJECT):
    runs = sorted(d for d in os.listdir(ABL_PROJECT) if d.startswith("ablation"))
    print("runs ->", ABL_PROJECT)
    print("existing runs:", runs or "(none - fresh start)")
    t = os.path.join(ABL_PROJECT, ".write_test")
    try:
        open(t, "w").write("ok"); os.remove(t); print("WRITABLE OK")
    except Exception as e:
        print("NOT WRITABLE:", e, "-- resume needs write access")
else:
    print("NOT FOUND:", ABL_PROJECT)
    print("Either you are on the wrong Google account, or the folder is")
    print("'Shared with me' and needs: right-click -> Organize ->")
    print("Add shortcut to Drive -> My Drive. Colab only mounts My Drive.")
    print("Creating it fresh below - ONLY correct if this is a first run.")
    os.makedirs(ABL_PROJECT, exist_ok=True)

In [ ]:
import os
REPO_URL = "https://github.com/FooJames/DEEPLRN_Group2.git"
if not os.path.isdir("DEEPLRN_Group2"):
    !git clone $REPO_URL
else:
    !cd DEEPLRN_Group2 && git pull
%cd DEEPLRN_Group2

In [ ]:
from google.colab import userdata
import os
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("key loaded:", bool(os.environ.get("ROBOFLOW_API_KEY")))

In [ ]:
# Both datasets + the two mandatory post-download fixes
!python scripts/download_data.py --child-version 3 --hazard-version 1
!python scripts/fix_data_yaml.py data/child/data.yaml data/hazard/data.yaml
!python scripts/normalize_child_labels.py data/child

## Session A - hazard (~2.3 h)

Uses the Phase 2 winner (`lr0=8.8e-4, box=8.14, cls=0.75, dfl=1.06`) held
fixed at every resolution. Re-run this cell to resume.

In [ ]:
!python scripts/ablation_imgsz.py --model hazard --data data/hazard/data.yaml     --epochs 30 --project "$ABL_PROJECT"

In [ ]:
!zip -r ablation_imgsz_hazard.zip results/metrics/ablation_imgsz_hazard.csv "$ABL_PROJECT"/ablation_imgsz_hazard_*
!unzip -l ablation_imgsz_hazard.zip | tail -5
from google.colab import files
files.download("ablation_imgsz_hazard.zip")

## Session B - child (~2.0 h)

Child was never tuned, so this runs ultralytics defaults — stated as an
asymmetry in the write-up. Within the model, all three runs are identical
except resolution, so the comparison is still clean.

In [ ]:
!python scripts/ablation_imgsz.py --model child --data data/child/data.yaml     --epochs 30 --project "$ABL_PROJECT"

In [ ]:
!zip -r ablation_imgsz_child.zip results/metrics/ablation_imgsz_child.csv "$ABL_PROJECT"/ablation_imgsz_child_*
!unzip -l ablation_imgsz_child.zip | tail -5
from google.colab import files
files.download("ablation_imgsz_child.zip")

## Results

In [ ]:
import pandas as pd, os
cols = ["imgsz", "mAP50", "mAP50_95", "train_min", "infer_ms"]
for m in ("hazard", "child"):
    p = f"results/metrics/ablation_imgsz_{m}.csv"
    if os.path.isfile(p):
        print(f"--- {m} ---")
        print(pd.read_csv(p)[cols].to_string(index=False))
        print()
print("baselines @640/100ep: child mAP50=0.947 | hazard mAP50=0.566")

### Notes
- `infer_ms` per resolution feeds the computational-cost comparison the
  proposal promises (two models per frame vs one).
- The two detectors are independent, so they may legitimately end up at
  **different** resolutions (e.g. child 416, hazard 832).
- Val split only; the test split stays untouched until the end.

---

## Session C - child re-run at the correct learning rate (~2 h)

**Why:** Phase 3b found the child detector was undertrained here. Child was
never tuned, so Sessions A/B ran it at ultralytics' default `lr0=0.01`.
At the auto-derived rate for a single-class dataset (`lr0=0.002`), with
everything else identical, child gains **+0.021 mAP50 / +0.050 mAP50-95**.

The resolution *ranking* from Session B is not invalid (all three sizes
used the same rate), but it was measured at a handicapped operating point.
This re-run confirms whether 640 still wins at the correct rate.

`--tag lr002` keeps this separate from the original run: results go to
`ablation_imgsz_child_lr002.csv` and runs to
`ablation_imgsz_child_lr002_<size>/`, so nothing overwrites Session B and
the earlier runs are not mistaken for this one.

In [ ]:
!python scripts/ablation_imgsz.py --model child --data data/child/data.yaml     --epochs 30 --lr0 0.002 --tag lr002 --project "$ABL_PROJECT"

In [ ]:
!zip -r ablation_imgsz_child_lr002.zip results/metrics/ablation_imgsz_child_lr002.csv "$ABL_PROJECT"/ablation_imgsz_child_lr002_*
!unzip -l ablation_imgsz_child_lr002.zip | tail -5
from google.colab import files
files.download("ablation_imgsz_child_lr002.zip")

In [ ]:
# Old vs new learning rate, side by side
import pandas as pd, os
old = "results/metrics/ablation_imgsz_child.csv"
new = "results/metrics/ablation_imgsz_child_lr002.csv"
if os.path.isfile(new):
    a = pd.read_csv(old)[["imgsz","mAP50","mAP50_95"]].set_index("imgsz")
    b = pd.read_csv(new)[["imgsz","mAP50","mAP50_95"]].set_index("imgsz")
    cmp = a.join(b, lsuffix="_lr0.01", rsuffix="_lr0.002")
    cmp["d_mAP50"] = cmp["mAP50_lr0.002"] - cmp["mAP50_lr0.01"]
    cmp["d_mAP50_95"] = cmp["mAP50_95_lr0.002"] - cmp["mAP50_95_lr0.01"]
    print(cmp.round(4).to_string())
    print("
best imgsz at lr0=0.002:", cmp["mAP50_lr0.002"].idxmax(),
          "| at lr0=0.01:", cmp["mAP50_lr0.01"].idxmax())
else:
    print("re-run not finished yet")